# 關鍵點偵測模型訓練

## 載入資料

In [ ]:
import pandas as pd
import os
import cv2
import matplotlib.pyplot as plt
import numpy as np
import json
from shared.functions import *

In [ ]:
# Function to get keypoints for a specified image filename
def get_keypoints_for_image(filename, keypoints_data_src):
    # Load your VIA project JSON file content (as a string or from a file)
    with open(keypoints_data_src, 'r') as f:
        via_data = json.load(f)

    # Mapping from file IDs to filenames
    file_id_to_name = {fid: fdata.get('fname', '') for fid, fdata in via_data['file'].items()}

    # Find associated file ID for the given filename
    file_ids = [fid for fid, name in file_id_to_name.items() if name == filename]
    if not file_ids:
        return None  # No matching file found
    keypoints = []
    for meta in via_data['metadata'].values():
        if meta['vid'] == file_ids[0]:
            xy = meta.get('xy', [])
            if len(xy) == 3:
                # xy format: [1, x, y] — take x, y
                keypoints.append((xy[1], xy[2]))
    return keypoints

def get_image(filename):
    img = cv2.imread(filename)
    return img

from typing import List, Tuple
import numpy as np
# If using cv2/image libraries, you can import cv2 as well for actual resizing.

def resize_image_and_keypoints(
    image: np.ndarray,
    keypoints: List[Tuple[float, float]],
    new_width: int,
    new_height: int
) -> Tuple[np.ndarray, List[Tuple[float, float]]]:
    """
    Resize input image and update keypoints to match the new dimensions.
    
    Args:
        image (np.ndarray): Original image array.
        keypoints (list of (x, y)): List of keypoints (floats).
        new_width (int): Target image width.
        new_height (int): Target image height.

    Returns:
        resized_image (np.ndarray): The resized image array.
        resized_keypoints (list of (x, y)): Scaled-updated keypoints.
    """
    orig_height, orig_width = image.shape[:2]
    scale_x = new_width / orig_width
    scale_y = new_height / orig_height

    # If OpenCV is available:
    # import cv2
    resized_image = cv2.resize(image, (new_width, new_height), interpolation=cv2.INTER_AREA)
    
    # Rescale keypoints
    resized_keypoints = [(x * scale_x, y * scale_y) for (x, y) in keypoints]
    return resized_image, resized_keypoints

In [ ]:
from ultralytics import YOLO

# 模型與圖片路徑
model_path = "models/yolo_finetuned/best.pt"

# 只保留這些類別 ID（根據 data.yaml 順序）
allowed_classes = [1]  # 只要床單

# 載入模型
yolo_model_finetuned = YOLO(model_path)

In [ ]:
import zipfile

img_arr = []
keypoints_img_arr = []
rgb_img_arr = []
file_paths = [] 

src_obj3 = {
    "keypoints_data_src": "via_proj/via_project_22Aug2025_16h07m06s.json",
    "image_path": "RGB-images-jpg/",
}

src_obj1 = {
    "keypoints_data_src": "via_proj/via_project_13Aug2025_15h48m06s.json",
    "realsense_path": "realsense/realsense_data/realsense_camera/",
    "realsense_depth_path": "realsense/realsense_data/depth_scaled/",
    "realsense_mask_path": "realsense/realsense_data/polygon_mask/",
    "ignore_list":[
        "color_1755068936.png",
        "color_1755069393.png",
        "color_1755069468.png",
        "color_1755069604.png",
    ]
}

src_obj2 = {
    "keypoints_data_src": "via_proj/via_project_21Aug2025_10h40m33s.json",
    "realsense_path": "realsense/realsense_data2/realsense_camera/",
    "realsense_depth_path": "realsense/realsense_data2/realsense_camera/",
    "realsense_mask_path": "realsense/realsense_data2/polygon_mask/",
    "ignore_list":[
        "color_1755739126.png",
        "color_1755739200.png",
        "color_1755739258.png",
        "color_1755740502.png",
        "color_1755740533.png",
        "color_1755740550.png",
        "color_1755740563.png",
        "color_1755740619.png",
        "color_1755740786.png",
        "color_1755740813.png",
        "color_1755740831.png",
        "color_1755740850.png",
        "color_1755740868.png",
        "color_1755741074.png",
        "color_1755741088.png",
        "color_1755741219.png",
    ]
}

def generate_data_arr(src_obj):
    keypoints_data_src = src_obj["keypoints_data_src"]
    # realsense_path = src_obj["realsense_path"]
    # realsense_depth_path = src_obj["realsense_depth_path"]
    # realsense_mask_path = src_obj["realsense_mask_path"]
    # ignore_list = src_obj["ignore_list"]
    image_path = src_obj["image_path"]
    for f in os.listdir(image_path):
        # if f[:6] == "color_":
            # fnumber = f[6:]; fnumber = fnumber[:-4]
            # depth_f = "depth_raw_" + fnumber + ".npy"
            # color_f = "color_" + fnumber + ".png"
            # ignore messy examples
            # if color_f in ignore_list:
            #     continue
            # Example usage:
            # depth_map = np.load(realsense_depth_path + depth_f)
            # img = depth_map_to_image(depth_map)
            # img =  cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
            # Now you can save with cv2.imwrite or display with OpenCV/Matplotlib
            # color_img = get_image(realsense_path+color_f)
            # mask = cv2.imread(realsense_mask_path + color_f, cv2.IMREAD_GRAYSCALE)  # shape: (H, W), dtype: uint8

            # Optionally, binarize: ensure strictly 0 (background) and 255 (foreground)
        
            # _, mask = cv2.threshold(mask, 127, 255, cv2.THRESH_BINARY)
        color_f = f
        img = cv2.imread(image_path + f)
        color_img = img.copy()
        # only use example where mask is detected.
        
        # mask the depth image
        orig_keypoints = get_keypoints_for_image(color_f, keypoints_data_src)

        img, keypoints = resize_image_and_keypoints(img, orig_keypoints, 128, 128)

        mask = extract_mask_compare(img, yolo_model_finetuned, allowed_classes)
        if np.sum(mask) > 0:
            img[mask==0] = 0

            color_img, keypoints = resize_image_and_keypoints(color_img, orig_keypoints, 128, 128)
            keypoints = [[kp[1], kp[0]] for kp in keypoints]
            
            #kp image
            kp_img = np.zeros((128, 128))
            for kp in keypoints:
                kp_img[int(kp[0]), int(kp[1])] = 1
            file_paths.append(image_path + f)
            img_arr.append(img)
            rgb_img_arr.append(color_img)
            keypoints_img_arr.append(kp_img)

In [ ]:
orig_hw = get_image("RGB-images-jpg/IMG_1305.jpg").shape[:2]


# generate the data arr
# generate_data_arr(src_obj1)
# generate_data_arr(src_obj2)
generate_data_arr(src_obj3)

img_arr = np.array(img_arr)
rgb_img_arr = np.array(rgb_img_arr)
keypoints_img_arr = np.array(keypoints_img_arr)

In [ ]:
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import torchvision.transforms.functional as TF
import random

class KeypointDataset(Dataset):
    def __init__(self, images, rgb_images, keypoints, file_paths, transform=None):
        self.images = images.astype(np.float32)
        self.rgb_images = rgb_images.astype(np.float32)
        self.keypoints = keypoints.astype(np.float32)
        self.file_paths = file_paths
        self.transform = transform

    def __len__(self):  # Add this method!
        return len(self.images)

    def __getitem__(self, idx):
        img = self.images[idx]
        rgb_img = self.rgb_images[idx]
        kp = self.keypoints[idx]
        img = np.transpose(img, (2, 0, 1))
        rgb_img = np.transpose(rgb_img, (2, 0, 1))
        sample = {'image': torch.from_numpy(img), 'keypoints': torch.from_numpy(kp), 'rgb_image': torch.from_numpy(rgb_img), 'file_path': self.file_paths[idx]}
        if self.transform:
            sample = self.transform(sample)
        return sample

class RandomRotateFlip:
    """
    Randomly applies:
    - A rotation by any angle in [0, 360)
    - Optionally, a horizontal flip with 50% chance after rotation
    """
    def __call__(self, sample):
        image, rgb_image, keypoints, file_path = sample['image'], sample["rgb_image"], sample['keypoints'], sample['file_path']
        # image: (C, H, W)
        # keypoints: (N, H, W) or (H, W)

        # --- Random rotation ---
        angle = random.uniform(0, 360)
        image = TF.rotate(image, angle, interpolation=TF.InterpolationMode.BILINEAR)
        rgb_image = TF.rotate(rgb_image, angle, interpolation=TF.InterpolationMode.BILINEAR)
        # For keypoints as heatmaps, use same rotate (assume keypoints is Tensor [N,H,W] or [H,W])
        # If N, treat each as a channel
        if keypoints.ndim == 3:
            keypoints = TF.rotate(keypoints, angle, interpolation=TF.InterpolationMode.BILINEAR)
        else:
            keypoints = TF.rotate(keypoints.unsqueeze(0), angle, interpolation=TF.InterpolationMode.BILINEAR).squeeze(0)

        # --- Random flip after rotation ---
        if random.random() < 0.5:
            image = TF.hflip(image)
            rgb_image = TF.hflip(rgb_image)
            keypoints = TF.hflip(keypoints)
        if random.random() < 0.5:
            image = TF.vflip(image)
            rgb_image = TF.vflip(rgb_image)
            keypoints = TF.vflip(keypoints)

        return {'image': image, 'rgb_image': rgb_image, 'keypoints': keypoints, 'file_path': file_path}

class SmartAugmentation:
    """
    Smart augmentation that adapts based on image characteristics
    """
    def __init__(self, p=0.3):
        self.p = p
    
    def __call__(self, sample):
        image, rgb_image, keypoints, file_path = sample['image'], sample["rgb_image"], sample['keypoints'], sample['file_path']
        
        # Calculate image statistics to make informed decisions
        img_mean = image.mean()
        img_std = image.std()
        
        # Only apply brightness/contrast if image is not too dark or too bright
        if 0.1 < img_mean < 0.9:  # Only augment if image is in reasonable brightness range
            if random.random() < self.p:
                brightness_factor = random.uniform(0.95, 1.05)  # Very conservative
                image = TF.adjust_brightness(image, brightness_factor)
                rgb_image = TF.adjust_brightness(rgb_image, brightness_factor)
            
            if random.random() < self.p:
                contrast_factor = random.uniform(0.95, 1.05)  # Very conservative
                image = TF.adjust_contrast(image, contrast_factor)
                rgb_image = TF.adjust_contrast(rgb_image, contrast_factor)
        
        # Add very small noise only if image has good contrast
        if img_std > 0.1 and random.random() < self.p:
            noise_std = random.uniform(0.003, 0.01)  # Very small noise
            image = image + torch.randn_like(image) * noise_std
            rgb_image = rgb_image + torch.randn_like(rgb_image) * noise_std
        
        # Geometric transformations (always safe)
        angle = random.uniform(0, 360)
        image = TF.rotate(image, angle, interpolation=TF.InterpolationMode.BILINEAR)
        rgb_image = TF.rotate(rgb_image, angle, interpolation=TF.InterpolationMode.BILINEAR)
        
        if keypoints.ndim == 3:
            keypoints = TF.rotate(keypoints, angle, interpolation=TF.InterpolationMode.BILINEAR)
        else:
            keypoints = TF.rotate(keypoints.unsqueeze(0), angle, interpolation=TF.InterpolationMode.BILINEAR).squeeze(0)

        if random.random() < 0.5:
            image = TF.hflip(image)
            rgb_image = TF.hflip(rgb_image)
            keypoints = TF.hflip(keypoints)
        if random.random() < 0.5:
            image = TF.vflip(image)
            rgb_image = TF.vflip(rgb_image)
            keypoints = TF.vflip(keypoints)

        return {'image': image, 'rgb_image': rgb_image, 'keypoints': keypoints, 'file_path': file_path}

In [ ]:
rotate_transform = SmartAugmentation()

# Create the full dataset without transform
full_dataset = KeypointDataset(img_arr, rgb_img_arr, keypoints_img_arr, file_paths, transform=None)

# Split indices for train/test
generator = torch.Generator().manual_seed(42)
train_size = int(0.9 * len(full_dataset))
test_size = len(full_dataset) - train_size
train_indices, test_indices = torch.utils.data.random_split(range(len(full_dataset)), [train_size, test_size], generator=generator)

# Create train and test datasets with/without transform
train_dataset = torch.utils.data.Subset(KeypointDataset(img_arr, rgb_img_arr, keypoints_img_arr, file_paths, transform=rotate_transform), train_indices)
test_dataset = torch.utils.data.Subset(KeypointDataset(img_arr, rgb_img_arr, keypoints_img_arr, file_paths, transform=None), test_indices)

trainloader = DataLoader(train_dataset, batch_size=32, shuffle=True, pin_memory=True)
testloader = DataLoader(test_dataset, batch_size=32, shuffle=False, pin_memory=True)

## 測試關鍵點跟圖片的正確性

In [ ]:
print(len(full_dataset))

In [ ]:
pair = train_dataset.__getitem__(7)
img = pair["image"].numpy().copy() / 255
img = np.transpose(img, (1, 2, 0))
img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)

rgb_img = pair["rgb_image"].numpy().copy() / 255
rgb_img = np.transpose(rgb_img, (1, 2, 0))
rgb_img = cv2.cvtColor(rgb_img, cv2.COLOR_RGB2BGR)

kp = pair["keypoints"].numpy()
print(img.shape, kp.shape)
for i in range(kp.shape[0]):
    for j in range(kp.shape[1]):
        if kp[i,j] > 0.1:
            cv2.circle(img, (j, i), 1, (255,0,0), -1)
plt.imshow(img)
plt.show()
plt.imshow(rgb_img)
plt.show()

## 訓練迴圈

In [ ]:
load_model = False

In [ ]:
from models.utils import *

In [ ]:
def spatial_klloss(pred_map, target_map, eps=1e-8):
    # pred_map: after spatial softmax, (B, 1, H, W)
    # target_map: one-hot or few-hot, (B, H, W)
    B, _, H, W = pred_map.shape
    pred = pred_map.view(B, -1) + eps  # avoid log(0)
    target = target_map.view(B, -1) + eps
    pred_log = pred.log()
    target = target / target.sum(dim=1, keepdim=True)  # ensure sum-to-1; safe for multi-keypoint
    return (target * (target.log() - pred_log)).sum(dim=1).mean()

def kl_heatmap_loss(pred_hm, gt_hm, mask=None, reduction='mean'):
    # pred_hm: (B, 1, H, W)
    # gt_hm:   (B, 1, H, W)
    # mask:    (B, 1, H, W) or None

    eps = 1e-8

    # Force positive
    pred_probs = pred_hm.clamp(min=eps)
    gt_probs = gt_hm.clamp(min=eps)

    # Optionally apply mask
    if mask is not None:
        pred_probs = pred_probs * mask
        gt_probs = gt_probs * mask

    # Sum per sample
    pred_sum = pred_probs.sum(dim=(2, 3), keepdim=True)
    gt_sum = gt_probs.sum(dim=(2, 3), keepdim=True)

    # Identify gt_hm slices that are all zeros (or close enough)
    gt_zero_mask = (gt_sum < eps).squeeze(1).squeeze(1)  # (B,) boolean: True means skip or zero out

    # Safe normalization (avoids divide by zero)
    pred_probs = pred_probs / pred_sum.clamp(min=eps)
    gt_probs = torch.where(gt_sum < eps, torch.zeros_like(gt_probs), gt_probs / gt_sum.clamp(min=eps))

    # Compute KL divergence per sample
    log_pred = pred_probs.log()
    kl_div = F.kl_div(log_pred, gt_probs, reduction='none').sum(dim=(2, 3))  # shape (B,1)
    kl_div = kl_div.squeeze(1)  # (B,)

    # For samples where gt_hm is all zeros, set loss to 0 (no supervision there)
    kl_div = kl_div.masked_fill(gt_zero_mask, 0.)

    if reduction == 'mean':
        num = (~gt_zero_mask).float().sum().clamp(min=1)
        return kl_div.sum() / num
    elif reduction == 'sum':
        return kl_div.sum()
    else:
        return kl_div
    
def batch_gaussian_blur(x, kernel_size=5, sigma=2.0):
    """
    Apply Gaussian blur to a batch of heatmaps and normalize each so the max is 1.
    Args:
        x: Tensor [B, H, W] or [B, 1, H, W]
    Returns:
        Tensor with same shape, blurred and with peak 1 per sample
    """
    unsqueeze = False
    if x.dim() == 3:  # [B, H, W]
        x = x.unsqueeze(1)
        unsqueeze = True
    
    blurred = TF.gaussian_blur(x, kernel_size=[kernel_size, kernel_size], sigma=[sigma, sigma])
    max_vals = blurred.amax(dim=[2, 3], keepdim=True)
    max_vals[max_vals == 0] = 1.0  # Avoid division by zero
    normalized = blurred / max_vals

    if unsqueeze:
        normalized = normalized.squeeze(1)
    return normalized

def batch_entropy(pred_heatmaps):
    """
    pred_heatmaps: [B, C, H, W]
    Returns: [B] entropy per image
    """
    # Flatten spatial dimensions (and optionally channels) for softmax
    B, C, H, W = pred_heatmaps.shape
    flat = pred_heatmaps.view(B, -1)                # [B, C*H*W]
    probs = torch.softmax(flat, dim=1)              # normalize to sum=1 per image
    entropy = -(probs * torch.log(probs + 1e-8)).sum(dim=1)  # [B]
    return entropy

In [ ]:
import torch.optim as optim
import torch.nn as nn
from models.yolo_vit import HybridKeypointNet
from ultralytics import YOLO
import time

# optimization
from torch.amp import autocast, GradScaler
torch.backends.cudnn.benchmark = True
scaler = GradScaler()

# create device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# loss
loss_fn = nn.BCEWithLogitsLoss()

# yolo vit
yolo_model = YOLO('yolo11l-pose.pt')
backbone_seq = yolo_model.model.model[:12]
backbone = YoloBackbone(backbone_seq, selected_indices=[0,1,2,3,4,5,6,7,8,9,10,11])
input_dummy = torch.randn(1, 3, 128, 128)
with torch.no_grad():
    feats = backbone(input_dummy)
in_channels_list = [f.shape[1] for f in feats]
keypoint_net = HybridKeypointNet(backbone, in_channels_list)
model = keypoint_net
# Freeze all layers first for safety
for param in model.backbone.parameters():
    param.requires_grad = False
# for param in model.encoder.parameters():
#     param.requires_grad = False
# for param in model.fusion.parameters():
#     param.requires_grad = False

model = model.to(device)

compiled_model = torch.compile(model)
# load pretrained model
compiled_model.load_state_dict(torch.load('models/keypoint_model_vit.pth', map_location=device))
compiled_model.eval()

def mixup_data(x, y, alpha=0.2):
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1

    batch_size = x.size()[0]
    index = torch.randperm(batch_size).to(x.device)

    mixed_x = lam * x + (1 - lam) * x[index, :]
    mixed_y = lam * y + (1 - lam) * y[index, :]
    
    return mixed_x, mixed_y

from torch.optim.lr_scheduler import StepLR

if not load_model:
    optimizer = optim.AdamW(compiled_model.parameters(), lr=1e-4, weight_decay=1e-4)
    scheduler = StepLR(optimizer, step_size=30, gamma=0.5)  # Reduce LR every 30 epochs

    for epoch in range(300):
        time_start = time.time()
        compiled_model.train()
        running_loss = 0.0

        for batch in trainloader:
            images = batch["image"].to(device)
            keypoints = batch["keypoints"].to(device)
            optimizer.zero_grad()

            with autocast("cuda", dtype=torch.float16):
                outputs = compiled_model(images)
                keypoints_blur = batch_gaussian_blur(keypoints, kernel_size=31, sigma=3)
                loss = kl_heatmap_loss(outputs, keypoints_blur.unsqueeze(1))

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            running_loss += loss.item() * images.size(0)
        
        train_loss = running_loss / len(train_dataset)
        print(f'Epoch {epoch+1}: Loss {train_loss:.4f}, LR: {scheduler.get_last_lr()[0]:.2e}')
        
        # No metric needed for StepLR
        scheduler.step()

    torch.save(compiled_model.state_dict(), 'models/keypoint_model_vit_depth.pth')
else:
    
    compiled_model.load_state_dict(torch.load('models/keypoint_model_vit_depth.pth', map_location=device))
    compiled_model.eval()

## 模型結果分析

In [ ]:
orig_hw = get_image("RGB-images-jpg/IMG_1305.jpg").shape[:2]

In [ ]:
# Evaluate on the validation set
compiled_model.eval()
val_loss = 0.0
iter = 0
with torch.no_grad():
    for batch in testloader:
        images = batch['image'].to(device)
        rgb_images = batch['rgb_image'].to(device)
        keypoints = batch['keypoints'].to(device)
        file_paths = batch['file_path']  # Changed from rgb_origs
        with autocast("cuda", dtype=torch.float16): 
            outputs = compiled_model(images)
            keypoints_blur = batch_gaussian_blur(keypoints, kernel_size=31, sigma=3)
            loss = kl_heatmap_loss(outputs, keypoints_blur.unsqueeze(1))
        # render the predicted keypoints on the image
        for img, kp, rgb_img, file_path in zip(images.cpu().numpy(), outputs.cpu().numpy(), rgb_images.cpu().numpy(), file_paths):
            img = np.transpose(img, (1, 2, 0))
            # Load original image from file path
            rgb_img_orig = cv2.imread(file_path)
            # Convert BGR to RGB for consistency
            rgb_img_orig = cv2.cvtColor(rgb_img_orig, cv2.COLOR_BGR2RGB)
            # Resize to original dimensions
            rgb_img_orig = cv2.resize(rgb_img_orig, (orig_hw[1], orig_hw[0]))
            img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
            kp = kp[0,:,:]
            peaks = thresholded_locations(kp, 0.003)
            for p in peaks:
                i,j = p
                cv2.circle(rgb_img_orig, (int(j * orig_hw[1]/128), int(i * orig_hw[0]/128)), 30, (255,0,0), -1)
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            rgb_img_orig = cv2.cvtColor(rgb_img_orig, cv2.COLOR_RGB2BGR)
            cv2.imwrite(f'results/keypoints_{iter}.png', rgb_img_orig)
            iter += 1
        val_loss += loss.item() * images.size(0)
    print(f'Validation Loss: {val_loss / len(test_dataset):.4f}')

## 模型視覺化

In [ ]:
from torchview import draw_graph

# Suppose `model` is your nn.Module, and x is a sample input tensor
model_graph = draw_graph(model, input_data=torch.randn((8,3,128,128)), expand_nested=True)
model_graph.visual_graph.render(filename='architecture_full', format='png')
# or to view inline in a Jupyter notebook:
display(model_graph.visual_graph)